In [ ]:
#!/usr/bin/env python3

"""
Wildfire-centered temporal disturbance analysis.

For every wildfire pixel-year from 2000-2020, characterize the occurrence
of biotic disturbance (BT) and hotter drought (HD) from 10 years before
through 10 years after wildfire.

Analyses are performed separately for:
    1. Any disturbance thresholds
    2. Extreme disturbance thresholds

Important:
- Every wildfire pixel-year is treated as an event.
- A pixel that burns more than once contributes multiple wildfire events.
- Years outside 2000-2020 are treated as unobservable, NOT as zero disturbance.
- All calculations use the native 30-m data.
- Outputs are summaries for later visualization in R.
"""

from pathlib import Path
from collections import Counter, defaultdict
import csv

import numpy as np
import rasterio


################################################################################
# PATHS
################################################################################

PROJECT_DIR = Path(
    "/home/jovyan/data-store/forest-disturbance-stack-v3"
)

BINARY_DIR = (
    PROJECT_DIR /
    "data/derived/annual_stacks/binary"
)

FOREST_MASK = (
    PROJECT_DIR /
    "data/derived/resampled/forest_mask_30m_resampled.tif"
)

ECO_RASTER = (
    PROJECT_DIR /
    "data/derived/ecoregions/ecoregions_level3_30m.tif"
)

OUT_DIR = (
    PROJECT_DIR /
    "data/derived/disturbance_sequence/wildfire_context"
)

SUMMARY_DIR = (
    PROJECT_DIR /
    "data/derived/disturbance_sequence/summaries"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


################################################################################
# SETTINGS
################################################################################

YEARS = list(
    range(2000, 2021)
)

THRESHOLDS = [
    "any",
    "extreme"
]

# Annual binary stack:
# band 1 = wildfire
# band 2 = biotic disturbance
# band 3 = hotter drought
# band 4 = PDSI

WF_BAND = 1
BT_BAND = 2
HD_BAND = 3

MAX_LAG = 10

WINDOWS = [
    1,
    3,
    5
]

PIXEL_AREA_KM2 = 0.0009

NODATA_UINT8 = 255


################################################################################
# INPUT PATHS
################################################################################

def annual_stack_path(threshold, year):

    return (
        BINARY_DIR /
        f"annual_stack_{threshold}_{year}.tif"
    )


################################################################################
# VALIDATE INPUTS
################################################################################

print("Validating inputs...")

if not FOREST_MASK.exists():
    raise FileNotFoundError(FOREST_MASK)

if not ECO_RASTER.exists():
    raise FileNotFoundError(ECO_RASTER)

for threshold in THRESHOLDS:

    for year in YEARS:

        path = annual_stack_path(
            threshold,
            year
        )

        if not path.exists():
            raise FileNotFoundError(path)


################################################################################
# REFERENCE GRID
################################################################################

reference_path = annual_stack_path(
    "any",
    2000
)

with rasterio.open(reference_path) as ref:

    ref_width = ref.width
    ref_height = ref.height
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_profile = ref.profile.copy()


def check_alignment(path):

    with rasterio.open(path) as src:

        if (
            src.width != ref_width or
            src.height != ref_height or
            src.transform != ref_transform or
            src.crs != ref_crs
        ):

            raise ValueError(
                f"Raster alignment mismatch: {path}"
            )


check_alignment(FOREST_MASK)
check_alignment(ECO_RASTER)

for threshold in THRESHOLDS:
    for year in YEARS:
        check_alignment(
            annual_stack_path(
                threshold,
                year
            )
        )

print("All inputs are aligned.")


################################################################################
# OUTPUT RASTER PROFILE
################################################################################

frequency_profile = ref_profile.copy()

frequency_profile.update(
    count=1,
    dtype="uint8",
    nodata=NODATA_UINT8,
    compress="DEFLATE",
    predictor=2,
    tiled=True,
    BIGTIFF="YES"
)


################################################################################
# GLOBAL SUMMARY COUNTERS
################################################################################

# Relative-year analysis
#
# Key:
# threshold, context disturbance, relative year
#
# Values:
# number of wildfire pixel-events eligible for that relative year
# number experiencing context disturbance

relative_eligible = Counter()
relative_disturbed = Counter()

# Same thing by ecoregion

relative_eligible_eco = Counter()
relative_disturbed_eco = Counter()


# Window summaries
#
# We will accumulate distributions rather than just means.
#
# Key:
# threshold, context, direction, window, n_context_years

window_count_distribution = Counter()
window_count_distribution_eco = Counter()


# Presence within window
#
# Key:
# threshold, context, direction, window

window_eligible = Counter()
window_present = Counter()

window_eligible_eco = Counter()
window_present_eco = Counter()


# Same-year disturbance

same_year_eligible = Counter()
same_year_present = Counter()

same_year_eligible_eco = Counter()
same_year_present_eco = Counter()


# Nearest occurrence
#
# Key:
# threshold, context, direction, lag

nearest_lag_counts = Counter()
nearest_lag_counts_eco = Counter()


# Wildfire recurrence

wildfire_frequency_counts = {}
wildfire_frequency_counts_eco = {}


################################################################################
# PROCESS ANY AND EXTREME
################################################################################

for threshold in THRESHOLDS:

    print()
    print("=" * 80)
    print(
        f"PROCESSING {threshold.upper()} WILDFIRE CONTEXT"
    )
    print("=" * 80)


    ########################################################################
    # OPEN ANNUAL RASTERS
    ########################################################################

    annual_sources = {
        year: rasterio.open(
            annual_stack_path(
                threshold,
                year
            )
        )
        for year in YEARS
    }

    forest_src = rasterio.open(
        FOREST_MASK
    )

    eco_src = rasterio.open(
        ECO_RASTER
    )


    ########################################################################
    # WILDFIRE FREQUENCY OUTPUT
    ########################################################################

    threshold_out_dir = (
        OUT_DIR /
        threshold
    )

    threshold_out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    wildfire_frequency_path = (
        threshold_out_dir /
        "wildfire_frequency.tif"
    )

    wildfire_frequency_dst = rasterio.open(
        wildfire_frequency_path,
        "w",
        **frequency_profile
    )


    ########################################################################
    # RECURRENCE COUNTERS
    ########################################################################

    wf_frequency_counter = Counter()
    wf_frequency_eco_counter = Counter()


    ########################################################################
    # PROCESS BLOCKS
    ########################################################################

    block_number = 0

    for block_index, window in forest_src.block_windows(1):

        block_number += 1

        if block_number % 100 == 0:
            print(
                f"  Processed {block_number} blocks..."
            )


        ####################################################################
        # FOREST + ECOREGION
        ####################################################################

        forest = (
            forest_src.read(
                1,
                window=window
            ) == 1
        )

        eco = eco_src.read(
            1,
            window=window
        )


        ####################################################################
        # READ THE COMPLETE 21-YEAR TIME SERIES FOR THIS BLOCK
        ####################################################################

        wf_by_year = {}
        bt_by_year = {}
        hd_by_year = {}


        for year in YEARS:

            src = annual_sources[
                year
            ]

            wf_by_year[year] = (
                (
                    src.read(
                        WF_BAND,
                        window=window
                    ) > 0
                ) &
                forest
            )

            bt_by_year[year] = (
                (
                    src.read(
                        BT_BAND,
                        window=window
                    ) > 0
                ) &
                forest
            )

            hd_by_year[year] = (
                (
                    src.read(
                        HD_BAND,
                        window=window
                    ) > 0
                ) &
                forest
            )


        ####################################################################
        # WILDFIRE FREQUENCY
        ####################################################################

        wildfire_frequency = np.zeros(
            forest.shape,
            dtype=np.uint8
        )

        for year in YEARS:

            wildfire_frequency += (
                wf_by_year[year]
                .astype(np.uint8)
            )


        wildfire_frequency_out = (
            wildfire_frequency.copy()
        )

        wildfire_frequency_out[
            ~forest
        ] = NODATA_UINT8


        wildfire_frequency_dst.write(
            wildfire_frequency_out,
            1,
            window=window
        )


        ####################################################################
        # WESTWIDE WILDFIRE RECURRENCE
        ####################################################################

        frequencies, counts = np.unique(
            wildfire_frequency[
                forest
            ],
            return_counts=True
        )

        for frequency, count in zip(
            frequencies,
            counts
        ):

            wf_frequency_counter[
                int(frequency)
            ] += int(count)


        ####################################################################
        # ECOREGION WILDFIRE RECURRENCE
        ####################################################################

        eco_ids = np.unique(
            eco[forest]
        )

        eco_ids = [
            int(x)
            for x in eco_ids
            if x > 0
        ]


        for eco_id in eco_ids:

            eco_mask = (
                forest &
                (eco == eco_id)
            )

            frequencies, counts = np.unique(
                wildfire_frequency[
                    eco_mask
                ],
                return_counts=True
            )

            for frequency, count in zip(
                frequencies,
                counts
            ):

                wf_frequency_eco_counter[
                    (
                        eco_id,
                        int(frequency)
                    )
                ] += int(count)


        ####################################################################
        # WILDFIRE-CENTERED ANALYSIS
        ####################################################################

        for wf_year in YEARS:

            wf_event = (
                wf_by_year[
                    wf_year
                ]
            )

            n_wf_events = int(
                np.count_nonzero(
                    wf_event
                )
            )

            if n_wf_events == 0:
                continue


            ################################################################
            # ECOREGIONS CONTAINING THIS WILDFIRE EVENT
            ################################################################

            event_eco_ids = np.unique(
                eco[
                    wf_event
                ]
            )

            event_eco_ids = [
                int(x)
                for x in event_eco_ids
                if x > 0
            ]


            ################################################################
            # SAME-YEAR BT AND HD
            ################################################################

            for context_name, context_data in [
                ("BT", bt_by_year),
                ("HD", hd_by_year)
            ]:

                context_same_year = (
                    context_data[
                        wf_year
                    ]
                )


                same_year_eligible[
                    (
                        threshold,
                        context_name
                    )
                ] += n_wf_events


                same_year_present[
                    (
                        threshold,
                        context_name
                    )
                ] += int(
                    np.count_nonzero(
                        wf_event &
                        context_same_year
                    )
                )


                ############################################################
                # SAME-YEAR BY ECOREGION
                ############################################################

                for eco_id in event_eco_ids:

                    eco_event = (
                        wf_event &
                        (eco == eco_id)
                    )

                    n_eco_events = int(
                        np.count_nonzero(
                            eco_event
                        )
                    )

                    same_year_eligible_eco[
                        (
                            threshold,
                            eco_id,
                            context_name
                        )
                    ] += n_eco_events


                    same_year_present_eco[
                        (
                            threshold,
                            eco_id,
                            context_name
                        )
                    ] += int(
                        np.count_nonzero(
                            eco_event &
                            context_same_year
                        )
                    )


            ################################################################
            # RELATIVE YEAR: -10 THROUGH +10
            ################################################################

            for relative_year in range(
                -MAX_LAG,
                MAX_LAG + 1
            ):

                target_year = (
                    wf_year +
                    relative_year
                )

                # Outside observation period:
                # unobservable, not zero.

                if target_year not in YEARS:
                    continue


                for context_name, context_data in [
                    ("BT", bt_by_year),
                    ("HD", hd_by_year)
                ]:

                    context_event = (
                        context_data[
                            target_year
                        ]
                    )


                    relative_eligible[
                        (
                            threshold,
                            context_name,
                            relative_year
                        )
                    ] += n_wf_events


                    relative_disturbed[
                        (
                            threshold,
                            context_name,
                            relative_year
                        )
                    ] += int(
                        np.count_nonzero(
                            wf_event &
                            context_event
                        )
                    )


                    ########################################################
                    # ECOREGION RELATIVE-YEAR SUMMARY
                    ########################################################

                    for eco_id in event_eco_ids:

                        eco_event = (
                            wf_event &
                            (eco == eco_id)
                        )

                        n_eco_events = int(
                            np.count_nonzero(
                                eco_event
                            )
                        )


                        relative_eligible_eco[
                            (
                                threshold,
                                eco_id,
                                context_name,
                                relative_year
                            )
                        ] += n_eco_events


                        relative_disturbed_eco[
                            (
                                threshold,
                                eco_id,
                                context_name,
                                relative_year
                            )
                        ] += int(
                            np.count_nonzero(
                                eco_event &
                                context_event
                            )
                        )


            ################################################################
            # PRE/POST WINDOWS
            ################################################################

            for context_name, context_data in [
                ("BT", bt_by_year),
                ("HD", hd_by_year)
            ]:

                for window_size in WINDOWS:

                    for direction in [
                        "pre",
                        "post"
                    ]:


                        ####################################################
                        # DEFINE YEARS IN WINDOW
                        ####################################################

                        if direction == "pre":

                            requested_years = list(
                                range(
                                    wf_year - window_size,
                                    wf_year
                                )
                            )

                        else:

                            requested_years = list(
                                range(
                                    wf_year + 1,
                                    wf_year + window_size + 1
                                )
                            )


                        # Require the FULL requested window.
                        #
                        # Example:
                        # a 2002 wildfire does not contribute to the
                        # 5-year pre-fire analysis.

                        if not all(
                            year in YEARS
                            for year in requested_years
                        ):
                            continue


                        ####################################################
                        # NUMBER OF CONTEXT-DISTURBANCE YEARS
                        ####################################################

                        context_count = np.zeros(
                            forest.shape,
                            dtype=np.uint8
                        )


                        for year in requested_years:

                            context_count += (
                                context_data[
                                    year
                                ].astype(np.uint8)
                            )


                        event_counts = (
                            context_count[
                                wf_event
                            ]
                        )


                        ####################################################
                        # ELIGIBLE EVENTS
                        ####################################################

                        window_eligible[
                            (
                                threshold,
                                context_name,
                                direction,
                                window_size
                            )
                        ] += n_wf_events


                        ####################################################
                        # AT LEAST ONE CONTEXT EVENT
                        ####################################################

                        window_present[
                            (
                                threshold,
                                context_name,
                                direction,
                                window_size
                            )
                        ] += int(
                            np.count_nonzero(
                                event_counts > 0
                            )
                        )


                        ####################################################
                        # FULL COUNT DISTRIBUTION
                        ####################################################

                        values, counts = np.unique(
                            event_counts,
                            return_counts=True
                        )

                        for value, count in zip(
                            values,
                            counts
                        ):

                            window_count_distribution[
                                (
                                    threshold,
                                    context_name,
                                    direction,
                                    window_size,
                                    int(value)
                                )
                            ] += int(count)


                        ####################################################
                        # ECOREGION WINDOW SUMMARIES
                        ####################################################

                        for eco_id in event_eco_ids:

                            eco_event = (
                                wf_event &
                                (eco == eco_id)
                            )

                            n_eco_events = int(
                                np.count_nonzero(
                                    eco_event
                                )
                            )

                            eco_counts = (
                                context_count[
                                    eco_event
                                ]
                            )


                            window_eligible_eco[
                                (
                                    threshold,
                                    eco_id,
                                    context_name,
                                    direction,
                                    window_size
                                )
                            ] += n_eco_events


                            window_present_eco[
                                (
                                    threshold,
                                    eco_id,
                                    context_name,
                                    direction,
                                    window_size
                                )
                            ] += int(
                                np.count_nonzero(
                                    eco_counts > 0
                                )
                            )


                            values, counts = np.unique(
                                eco_counts,
                                return_counts=True
                            )

                            for value, count in zip(
                                values,
                                counts
                            ):

                                window_count_distribution_eco[
                                    (
                                        threshold,
                                        eco_id,
                                        context_name,
                                        direction,
                                        window_size,
                                        int(value)
                                    )
                                ] += int(count)


            ################################################################
            # NEAREST BT / HD BEFORE AND AFTER FIRE
            ################################################################

            for context_name, context_data in [
                ("BT", bt_by_year),
                ("HD", hd_by_year)
            ]:

                for direction in [
                    "pre",
                    "post"
                ]:

                    nearest = np.zeros(
                        forest.shape,
                        dtype=np.uint8
                    )


                    # Search from nearest to farthest.

                    for lag in range(
                        1,
                        MAX_LAG + 1
                    ):

                        if direction == "pre":
                            target_year = wf_year - lag

                        else:
                            target_year = wf_year + lag


                        if target_year not in YEARS:
                            continue


                        match = (
                            wf_event &
                            (nearest == 0) &
                            context_data[
                                target_year
                            ]
                        )

                        nearest[
                            match
                        ] = lag


                    ########################################################
                    # RECORD ONLY EVENTS WITH AN OBSERVED CONTEXT EVENT
                    ########################################################

                    observed = (
                        wf_event &
                        (nearest > 0)
                    )

                    values, counts = np.unique(
                        nearest[
                            observed
                        ],
                        return_counts=True
                    )


                    for lag, count in zip(
                        values,
                        counts
                    ):

                        nearest_lag_counts[
                            (
                                threshold,
                                context_name,
                                direction,
                                int(lag)
                            )
                        ] += int(count)


                    ########################################################
                    # ECOREGION NEAREST LAGS
                    ########################################################

                    for eco_id in event_eco_ids:

                        eco_observed = (
                            observed &
                            (eco == eco_id)
                        )

                        values, counts = np.unique(
                            nearest[
                                eco_observed
                            ],
                            return_counts=True
                        )


                        for lag, count in zip(
                            values,
                            counts
                        ):

                            nearest_lag_counts_eco[
                                (
                                    threshold,
                                    eco_id,
                                    context_name,
                                    direction,
                                    int(lag)
                                )
                            ] += int(count)


    ########################################################################
    # CLOSE FILES
    ########################################################################

    wildfire_frequency_dst.close()

    forest_src.close()
    eco_src.close()

    for src in annual_sources.values():
        src.close()


    wildfire_frequency_counts[
        threshold
    ] = wf_frequency_counter

    wildfire_frequency_counts_eco[
        threshold
    ] = wf_frequency_eco_counter


    print(
        f"Finished {threshold.upper()}."
    )


################################################################################
# WRITE RELATIVE-YEAR WESTWIDE CSV
################################################################################

relative_file = (
    SUMMARY_DIR /
    "wildfire_context_relative_year.csv"
)

with open(
    relative_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "relative_year",
        "wildfire_pixel_events_eligible",
        "context_pixel_events",
        "percent"
    ])


    for key in sorted(relative_eligible):

        threshold, context, relative_year = key

        eligible = relative_eligible[key]

        disturbed = relative_disturbed[key]

        percent = (
            disturbed /
            eligible *
            100
            if eligible > 0
            else np.nan
        )


        writer.writerow([
            threshold,
            context,
            relative_year,
            eligible,
            disturbed,
            percent
        ])


################################################################################
# WRITE RELATIVE-YEAR ECOREGION CSV
################################################################################

relative_eco_file = (
    SUMMARY_DIR /
    "wildfire_context_relative_year_ecoregions.csv"
)

with open(
    relative_eco_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "ecoregion_id",
        "context_disturbance",
        "relative_year",
        "wildfire_pixel_events_eligible",
        "context_pixel_events",
        "percent"
    ])


    for key in sorted(
        relative_eligible_eco
    ):

        (
            threshold,
            eco_id,
            context,
            relative_year
        ) = key

        eligible = (
            relative_eligible_eco[
                key
            ]
        )

        disturbed = (
            relative_disturbed_eco[
                key
            ]
        )

        percent = (
            disturbed /
            eligible *
            100
            if eligible > 0
            else np.nan
        )


        writer.writerow([
            threshold,
            eco_id,
            context,
            relative_year,
            eligible,
            disturbed,
            percent
        ])


################################################################################
# WRITE WINDOW SUMMARY
################################################################################

window_file = (
    SUMMARY_DIR /
    "wildfire_context_windows.csv"
)

with open(
    window_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "direction",
        "window_years",
        "wildfire_pixel_events_eligible",
        "pixel_events_with_context",
        "percent_with_context"
    ])


    for key in sorted(
        window_eligible
    ):

        (
            threshold,
            context,
            direction,
            window_size
        ) = key

        eligible = (
            window_eligible[key]
        )

        present = (
            window_present[key]
        )


        writer.writerow([
            threshold,
            context,
            direction,
            window_size,
            eligible,
            present,
            (
                present /
                eligible *
                100
                if eligible > 0
                else np.nan
            )
        ])


################################################################################
# WRITE WINDOW COUNT DISTRIBUTIONS
################################################################################

window_distribution_file = (
    SUMMARY_DIR /
    "wildfire_context_window_distributions.csv"
)

with open(
    window_distribution_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "direction",
        "window_years",
        "n_context_years",
        "wildfire_pixel_events"
    ])


    for key, count in sorted(
        window_count_distribution.items()
    ):

        (
            threshold,
            context,
            direction,
            window_size,
            n_context_years
        ) = key


        writer.writerow([
            threshold,
            context,
            direction,
            window_size,
            n_context_years,
            count
        ])


################################################################################
# WRITE ECOREGION WINDOW SUMMARY
################################################################################

window_eco_file = (
    SUMMARY_DIR /
    "wildfire_context_windows_ecoregions.csv"
)

with open(
    window_eco_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "ecoregion_id",
        "context_disturbance",
        "direction",
        "window_years",
        "wildfire_pixel_events_eligible",
        "pixel_events_with_context",
        "percent_with_context"
    ])


    for key in sorted(
        window_eligible_eco
    ):

        (
            threshold,
            eco_id,
            context,
            direction,
            window_size
        ) = key

        eligible = (
            window_eligible_eco[key]
        )

        present = (
            window_present_eco[key]
        )


        writer.writerow([
            threshold,
            eco_id,
            context,
            direction,
            window_size,
            eligible,
            present,
            (
                present /
                eligible *
                100
                if eligible > 0
                else np.nan
            )
        ])


################################################################################
# WRITE ECOREGION WINDOW COUNT DISTRIBUTIONS
################################################################################

window_distribution_eco_file = (
    SUMMARY_DIR /
    "wildfire_context_window_distributions_ecoregions.csv"
)

with open(
    window_distribution_eco_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "ecoregion_id",
        "context_disturbance",
        "direction",
        "window_years",
        "n_context_years",
        "wildfire_pixel_events"
    ])


    for key, count in sorted(
        window_count_distribution_eco.items()
    ):

        (
            threshold,
            eco_id,
            context,
            direction,
            window_size,
            n_context_years
        ) = key


        writer.writerow([
            threshold,
            eco_id,
            context,
            direction,
            window_size,
            n_context_years,
            count
        ])


################################################################################
# WRITE SAME-YEAR SUMMARY
################################################################################

same_year_file = (
    SUMMARY_DIR /
    "wildfire_context_same_year.csv"
)

with open(
    same_year_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "wildfire_pixel_events",
        "same_year_pixel_events",
        "percent_same_year"
    ])


    for key in sorted(
        same_year_eligible
    ):

        threshold, context = key

        eligible = (
            same_year_eligible[key]
        )

        present = (
            same_year_present[key]
        )


        writer.writerow([
            threshold,
            context,
            eligible,
            present,
            (
                present /
                eligible *
                100
                if eligible > 0
                else np.nan
            )
        ])


################################################################################
# WRITE NEAREST-LAG SUMMARY
################################################################################

nearest_file = (
    SUMMARY_DIR /
    "wildfire_context_nearest_lags.csv"
)

with open(
    nearest_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "direction",
        "lag_years",
        "wildfire_pixel_events"
    ])


    for key, count in sorted(
        nearest_lag_counts.items()
    ):

        (
            threshold,
            context,
            direction,
            lag
        ) = key


        writer.writerow([
            threshold,
            context,
            direction,
            lag,
            count
        ])


################################################################################
# WRITE WILDFIRE RECURRENCE SUMMARY
################################################################################

reburn_file = (
    SUMMARY_DIR /
    "wildfire_reburn_summary.csv"
)

with open(
    reburn_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "wildfire_year_count",
        "pixel_count",
        "area_km2"
    ])


    for threshold in THRESHOLDS:

        counter = (
            wildfire_frequency_counts[
                threshold
            ]
        )

        for frequency in sorted(
            counter
        ):

            count = counter[
                frequency
            ]

            writer.writerow([
                threshold,
                frequency,
                count,
                count * PIXEL_AREA_KM2
            ])


################################################################################
# FINISHED
################################################################################

print()
print("=" * 80)
print("WILDFIRE-CENTERED ANALYSIS COMPLETE")
print("=" * 80)

print()
print("Main outputs:")

for path in [
    relative_file,
    relative_eco_file,
    window_file,
    window_distribution_file,
    window_eco_file,
    window_distribution_eco_file,
    same_year_file,
    nearest_file,
    reburn_file
]:

    print(
        f"  {path}"
    )

Validating inputs...
All inputs are aligned.

PROCESSING ANY WILDFIRE CONTEXT
  Processed 100 blocks...
  Processed 200 blocks...
  Processed 300 blocks...
  Processed 400 blocks...
  Processed 500 blocks...
  Processed 600 blocks...
  Processed 700 blocks...
  Processed 800 blocks...
  Processed 900 blocks...
  Processed 1000 blocks...
  Processed 1100 blocks...
  Processed 1200 blocks...
  Processed 1300 blocks...
  Processed 1400 blocks...
  Processed 1500 blocks...
  Processed 1600 blocks...
  Processed 1700 blocks...
  Processed 1800 blocks...
  Processed 1900 blocks...
  Processed 2000 blocks...
  Processed 2100 blocks...
  Processed 2200 blocks...
  Processed 2300 blocks...
  Processed 2400 blocks...
  Processed 2500 blocks...
  Processed 2600 blocks...
  Processed 2700 blocks...
  Processed 2800 blocks...
  Processed 2900 blocks...
  Processed 3000 blocks...
  Processed 3100 blocks...
  Processed 3200 blocks...
  Processed 3300 blocks...
  Processed 3400 blocks...
  Processed 3